# Autonomous RAG (Self-Corrective & Critique-Driven RAG)

## Table of Contents
1. [What is Autonomous RAG?](#what-is-autonomous-rag)
2. [Why, When, and How to Use It](#why-when-and-how-to-use-it)
3. [Flowchart & State Machine](#flowchart--state-machine)
4. [Pros and Cons](#pros-and-cons)
5. [Real-World Case Study: Automated Research & Intelligence Analyst](#real-world-case-study-automated-research--intelligence-analyst)
6. [Building from Scratch in LangGraph](#building-from-scratch-in-langgraph)

---

## What is Autonomous RAG?

**Autonomous RAG** (often referred to as Self-RAG or Critique-Loop RAG) is a highly autonomous, agentic retrieval-augmented generation pattern where the agent does not follow a fixed routing pathway. Instead, the agent autonomously decides:
1. **Whether retrieval is even necessary** to answer the user request.
2. **Which retrieval tool** is best suited (e.g. Vector DB, internal documents, external web APIs).
3. **If the retrieved documents contain sufficient information** or if it needs to dynamically fetch more.
4. **Whether the generated response is fully grounded** in the retrieved context (checking for hallucinations).
5. **Whether the response is completely satisfying** or needs revision.

By placing retrieval and generation inside a self-corrective critique loop, the LLM continuously evaluates its own outputs, query parameters, and context quality, mimicking the workflow of a human researcher.

---

## Why, When, and How to Use It

### Why Use Autonomous RAG?
Traditional RAG and even Adaptive RAG rely on fixed routing pipelines (e.g., if query classification is X, do Y). However, complex queries are often unpredictable. 
Autonomous RAG treats retrieval as a **tool-use decision** inside an iterative loop. It solves:
- **Retrieval Inadequacy**: If the first query retrieves poor data, the LLM detects the deficit, reformulates the search parameters, and tries again.
- **Hallucinations**: It explicitly grades the generation against the retrieved data before presenting the final answer to the user.
- **Redundant Processing**: The agent stops fetching and iterating immediately once it verifies the query is answered.

### When to Use It
- **Unstructured / Open-Ended Investigations**: Financial research, legal due diligence, or market analysis where the depth of search needed is unknown beforehand.
- **Complex Troubleshooting**: Technical support situations where a sequence of distinct lookups is required to diagnose a problem.
- **High-Accuracy Domains**: Where factual errors or hallucinations are completely unacceptable and self-grading is mandatory.

### How it Works (The Core Loop)
1. **Initial Assessment**: The LLM determines if it has enough knowledge to answer or if it needs tools.
2. **Autonomous Tool Call**: Executes retrieval tools (Vector DB search, web search, etc.).
3. **Document Relevance Critique**: Evaluates retrieved text. If inadequate, it generates a new search query and loops back to retrieval.
4. **Draft Generation**: Builds a candidate response.
5. **Hallucination & Completeness Critique**: Evaluates the candidate response. If it fails, the agent either regenerates or searches again.
6. **Conclude**: Once the critique criteria are met, it serves the validated final response.

---

## Flowchart & State Machine

### High-Level Autonomous Loop Workflow

```mermaid
graph TD
    UserQuery[User Query] --> Agent[Autonomous Agent]
    
    Agent -->|Decides Tool Call Needed| RetrievalTool[Vector DB / Web Search]
    Agent -->|Decides No Tools Needed| DirectAnswer[Direct Generation]
    
    RetrievalTool --> CritiqueDocs{Are Docs Relevant & Sufficient?}
    
    CritiqueDocs -->|No: Reformulate Query| Agent
    CritiqueDocs -->|Yes| GenerateDraft[Generate Draft Answer]
    
    GenerateDraft --> CritiqueDraft{Self-Critique & Grade Draft}
    
    CritiqueDraft -->|Hallucination Detected| Regenerate[Regenerate Draft]
    Regenerate --> CritiqueDraft
    
    CritiqueDraft -->|Incomplete/Missing Info| Agent
    CritiqueDraft -->|Validated & Grounded| FinalOutput[Final Verified Answer]
```

### Detailed LangGraph State Machine

```mermaid
stateDiagram-v2
    [*] --> CallAgent: Start
    
    CallAgent --> ExecuteTools: Agent Decides to Call Tools
    CallAgent --> FinalizeAnswer: Agent Decides to Respond Direct
    
    ExecuteTools --> CritiqueDocuments
    
    CritiqueDocuments --> CallAgent: Insufficient / Relevant Docs Need Revision
    CritiqueDocuments --> GenerateDraftAnswer: Docs are Sufficient
    
    GenerateDraftAnswer --> SelfCritique
    
    SelfCritique --> CallAgent: Incomplete (Retrieve More)
    SelfCritique --> GenerateDraftAnswer: Grounding Fail (Regenerate)
    SelfCritique --> FinalizeAnswer: Verified & Completed
    
    FinalizeAnswer --> [*]
```

---

## Pros and Cons

### Pros
- **Maximum Flexibility**: Handles arbitrary research tasks, multi-hop lookups, and unexpected search requirements.
- **Self-Healing**: Corrects its own mistakes, adjusts query keywords, and ignores irrelevant or noisy documents.
- **Unmatched Accuracy**: The rigorous double-pass grading (docs-to-query, generation-to-docs) ensures clean, factual answers.
- **Conversational Smoothness**: Handles normal conversation and deep information retrieval under a single unified state.

### Cons
- **Latency Spikes**: The agent can loop multiple times if it finds the documents insufficient, increasing response time.
- **High API Costs**: Iterative tool-use and self-reflection steps lead to multiple LLM calls per user prompt.
- **Complex Loop Termination**: Requires strict loop guards to prevent infinite retrieval cycles when matching documents don't exist.

---

## Real-World Case Study: Automated Research & Intelligence Analyst

### Scenario
An investment firm wants to automate their public equities research analyst. When a user asks a question about a company, the analyst needs to:
1. Determine what financial documents or public news articles it needs to read.
2. Formulate specific search parameters.
3. Review retrieved articles to ensure they contain hard numbers and verified facts (not rumors).
4. If an article doesn't answer the question (e.g. details about a specific executive transition), the analyst must dynamically formulate a secondary query (e.g. search for the executive's name) and read those documents.
5. Compile the research report, self-check for any factual discrepancies, and output the final verified dossier.

---

## Building from Scratch in LangGraph

We will implement this autonomous loop in LangGraph. The agent node uses a function-calling layout where it can output tool calls or choose to finalize the response. The state maintains an ongoing list of retrieved contexts, a critique log, and loop counters. The execution details are in the accompanying Python script.

## Complete Implementation Code

In [ ]:
"""
Autonomous RAG (Self-Corrective & Critique-Driven RAG) - LangGraph Implementation
===================================================================================

This script implements an Autonomous RAG pattern:
1. Autonomous Research Agent: LLM dynamically decides whether to call search tools or draft an answer.
2. Unified Mock Search Tool: Provides a retrieval mechanism over corporate history.
3. Self-Critique Node: Grades document relevance, checks for hallucinations, and evaluates answer completeness.
4. Autonomous Loop Control: Dynamically decides to search more, rewrite queries, or finalize.
"""

import os
from typing import Literal, List, Dict, Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# Load environment variables
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '.env'))
load_dotenv()

# ==============================================================================
# Dynamic Dual-Provider LLM Setup (OpenAI / Groq)
# ==============================================================================

if os.environ.get("OPENAI_API_KEY"):
    print("[INFO] Detected OpenAI API Key. Running with OpenAI...")
    from langchain_openai import ChatOpenAI
    agent_model = ChatOpenAI(model='gpt-4o', temperature=0)
    critique_model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
elif os.environ.get("GROQ_API_KEY"):
    print("[INFO] Detected Groq API Key. Running with Groq...")
    from langchain_groq import ChatGroq
    # Llama 3.3 70B is excellent for autonomous planning and structured output
    agent_model = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
    critique_model = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
else:
    raise ValueError("Missing credentials. Set OPENAI_API_KEY or GROQ_API_KEY in your .env file.")


# ==============================================================================
# Step 1: Define Mock Database (Multi-Hop Bridge Data)
# ==============================================================================

KNOWLEDGE_BASE = [
    {
        "keywords": ["apex dynamics", "ceo", "executive", "resignation"],
        "source": "Apex Dynamics Press Release (March 2026)",
        "content": (
            "Apex Dynamics Inc. announces that CEO John Doe has submitted his resignation "
            "effective April 30, 2026. The Board of Directors has appointed Sarah Jenkins "
            "as the new CEO, effective May 1, 2026. Jenkins joins from Zenith Corp."
        )
    },
    {
        "keywords": ["sarah jenkins", "zenith corp", "coo"],
        "source": "Corporate Registry - Sarah Jenkins Profile",
        "content": (
            "Sarah Jenkins served as Chief Operating Officer (COO) of Zenith Corp from 2021 to 2026. "
            "Prior to Zenith Corp, she was VP of Engineering at Autonomous Systems Ltd. She holds a "
            "Master's degree in Robotics from MIT."
        )
    },
    {
        "keywords": ["zenith corp", "patents", "intellectual property"],
        "source": "USPTO Database - Zenith Corp Patent Holdings",
        "content": (
            "Zenith Corp holds a total of 52 granted patents in autonomous vehicle guidance, "
            "lidar signal processing, and machine-learning-driven trajectory planning. Key patents "
            "include US-10822194-B2 covering neural network trajectory estimation."
        )
    }
]


# ==============================================================================
# Step 2: Define Retrieval Search Tool
# ==============================================================================

def search_corporate_records(query: str) -> str:
    """Queries corporate records and patent databases for matches."""
    query_lower = query.lower()
    matches = []
    print(f"   [Tool Execution] Searching corporate records for: '{query}'...")
    
    for doc in KNOWLEDGE_BASE:
        # Check if any query terms match keywords or content
        search_space = (" ".join(doc["keywords"]) + " " + doc["content"]).lower()
        if any(term in search_space for term in query_lower.split() if len(term) > 3):
            matches.append(f"Source: [{doc['source']}]\nContent: {doc['content']}")
            
    if not matches:
        return f"No matches found for search query: '{query}'"
    return "\n\n---\n\n".join(matches)


# ==============================================================================
# Step 3: Define Structured Agent and Critique Schemas
# ==============================================================================

# Schema for the Agent's next action
class AgentAction(BaseModel):
    """Represent the next step the autonomous research agent wants to take."""
    action: Literal["search", "draft_response"] = Field(
        description="Select 'search' to run a query against corporate records. Select 'draft_response' if you believe you have gathered all necessary information from search results to completely answer the user query."
    )
    search_query: str = Field(
        default="",
        description="The query string to search for. Must be populated if action is 'search'."
    )
    draft_answer: str = Field(
        default="",
        description="The candidate response to present to the critique panel. Must be populated if action is 'draft_response'."
    )
    reasoning: str = Field(description="Internal reasoning behind choosing this action.")


# Schema for the Critique Node
class CritiqueResult(BaseModel):
    """Result of self-critiquing the candidate answer."""
    status: Literal["approved", "needs_more_data", "needs_regeneration"] = Field(
        description="Choose 'approved' if the answer is grounded and complete. Choose 'needs_more_data' if there are unanswered aspects requiring new searches. Choose 'needs_regeneration' if the answer is hallucinated or logically flawed."
    )
    feedback: str = Field(description="Critique feedback detailing what is missing, hallucinated, or correct.")
    suggested_search: str = Field(
        default="",
        description="Optimized search query to fetch the missing details if status is 'needs_more_data'."
    )

# Bind structured output
structured_agent = agent_model.with_structured_output(AgentAction)
structured_critic = critique_model.with_structured_output(CritiqueResult)


# ==============================================================================
# Step 4: Define Graph State
# ==============================================================================

class AutonomousRAGState(MessagesState):
    """State tracks accumulated facts, tool actions, and critique scores."""
    accumulated_context: List[str]  # Facts collected across all search steps
    agent_action: str               # "search" or "draft_response"
    search_query: str               # Current search string
    candidate_answer: str           # The drafted answer
    critique_feedback: str          # Feedback from the critic
    loop_count: int                 # Loop counter to prevent infinite execution
    final_output: str               # Final verified output to return


# ==============================================================================
# Step 5: Implement Graph Nodes
# ==============================================================================

def autonomous_agent(state: AutonomousRAGState) -> dict:
    """The central reasoning engine. Decides to search or draft based on current knowledge."""
    user_query = state["messages"][-1].content
    context = "\n\n".join(state.get("accumulated_context", []))
    loop_count = state.get("loop_count", 0)
    
    print(f"\n[1. Agent Node] Analyzing state (Loop {loop_count})...")
    
    system_prompt = (
        "You are an Autonomous Research Analyst conducting a multi-hop investigation.\n"
        "Your task is to answer the user query completely and factually.\n\n"
        "RULES:\n"
        "1. Check the 'Accumulated Context' below. If you already have all the facts needed "
        "to answer the original query, select action='draft_response'.\n"
        "2. If you are missing facts, select action='search' and formulate a specific search_query. "
        "Avoid broad or generic queries; search for specific names, companies, or keywords.\n"
        "3. Do not make assumptions. Rely strictly on retrieved facts.\n\n"
        f"USER QUERY: {user_query}\n\n"
        f"ACCUMULATED CONTEXT:\n{context if context else 'None yet.'}"
    )
    
    decision = structured_agent.invoke([SystemMessage(content=system_prompt)])
    print(f"   Decided Action: {decision.action.upper()}")
    print(f"   Reasoning: {decision.reasoning}")
    
    return {
        "agent_action": decision.action,
        "search_query": decision.search_query,
        "candidate_answer": decision.draft_answer,
        "loop_count": loop_count
    }


def execute_search(state: AutonomousRAGState) -> dict:
    """Executes the search query formulated by the agent and appends results to context."""
    query = state["search_query"]
    context_list = state.get("accumulated_context", [])
    
    print(f"\n[2. Search Node] Executing retrieval...")
    search_result = search_corporate_records(query)
    
    # Store source results
    updated_context = context_list + [f"Query: '{query}' -> Results:\n{search_result}"]
    
    return {
        "accumulated_context": updated_context,
        "loop_count": state["loop_count"] + 1
    }


def self_critique(state: AutonomousRAGState) -> dict:
    """Critiques the candidate answer for grounding (hallucination) and completeness."""
    user_query = state["messages"][-1].content
    context = "\n\n".join(state.get("accumulated_context", []))
    candidate = state["candidate_answer"]
    
    print("\n[3. Critique Node] Evaluating candidate response...")
    
    prompt = (
        "You are an Independent Quality Assurance Critic. Evaluate the candidate response.\n\n"
        "CRITERIA:\n"
        "1. GROUNDING: Is every statement in the candidate response directly supported by the Accumulated Context? "
        "If there are hallucinated numbers, dates, names, or assumptions, mark status='needs_regeneration'.\n"
        "2. COMPLETENESS: Does the candidate response fully answer all parts of the User Query? "
        "If some parts of the question are unanswered due to missing context, mark status='needs_more_data' "
        "and suggest a specific query in 'suggested_search'.\n\n"
        f"User Query: {user_query}\n\n"
        f"Accumulated Context:\n{context}\n\n"
        f"Candidate Response:\n{candidate}"
    )
    
    critique = structured_critic.invoke([HumanMessage(content=prompt)])
    
    print(f"   Critique Verdict: {critique.status.upper()}")
    print(f"   Critique Feedback: {critique.feedback}")
    if critique.suggested_search:
        print(f"   Suggested Next Search: '{critique.suggested_search}'")
        
    # Update search query for the next loop if needed
    update = {
        "critique_feedback": critique.feedback,
        "agent_action": "search" if critique.status == "needs_more_data" else ("draft_response" if critique.status == "needs_regeneration" else "finalize"),
        "final_output": candidate if critique.status == "approved" else ""
    }
    
    if critique.status == "needs_more_data":
        update["search_query"] = critique.suggested_search
        
    return update


def finalize_answer(state: AutonomousRAGState) -> dict:
    """Concludes the graph, producing the finalized research output."""
    print("\n[4. Finalize Node] Preparing verified response.")
    ans = state["final_output"]
    return {
        "messages": [AIMessage(content=ans, name="ResearchAnalyst")]
    }


# ==============================================================================
# Step 6: Define Conditional Routers (State Machine Logic)
# ==============================================================================

def route_agent_decision(state: AutonomousRAGState) -> str:
    """Routes based on the agent's action decision."""
    if state["loop_count"] >= 6:
        print("\n[Loop Guard Alert] Maximum loops reached. Forcing final answer.")
        return "finalize"
        
    if state["agent_action"] == "search":
        return "execute_search"
    return "self_critique"

def route_critique_decision(state: AutonomousRAGState) -> str:
    """Routes based on the critic's grading result."""
    if state["agent_action"] == "finalize":
        return "finalize"
    elif state["agent_action"] == "search":
        # Loop back directly to execute search with the critic's suggested query
        return "execute_search"
    else:
        # Loop back to agent node for regeneration
        return "autonomous_agent"


# Build the graph
workflow = StateGraph(AutonomousRAGState)

# Add nodes
workflow.add_node("autonomous_agent", autonomous_agent)
workflow.add_node("execute_search", execute_search)
workflow.add_node("self_critique", self_critique)
workflow.add_node("finalize_answer", finalize_answer)

# Wire edges
workflow.add_edge(START, "autonomous_agent")

workflow.add_conditional_edges(
    "autonomous_agent",
    route_agent_decision,
    {
        "execute_search": "execute_search",
        "self_critique": "self_critique",
        "finalize": "finalize_answer"
    }
)

workflow.add_edge("execute_search", "autonomous_agent")

workflow.add_conditional_edges(
    "self_critique",
    route_critique_decision,
    {
        "finalize": "finalize_answer",
        "execute_search": "execute_search",
        "autonomous_agent": "autonomous_agent"
    }
)

workflow.add_edge("finalize_answer", END)

# Compile
compiled_graph = workflow.compile()


# ==============================================================================
# Step 7: Execution and Demonstration
# ==============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("AUTONOMOUS RAG (INTELLIGENCE DOSSIER ANALYST) - Start")
    print("=" * 80)
    
    # A complex multi-hop query requiring discovery of bridge entities and patent data
    user_query = (
        "Find out who the current CEO of Apex Dynamics is, "
        "where they worked previously, and how many patents that previous company holds."
    )
    
    print(f"Investigation Query:\n\"{user_query}\"\n")
    
    inputs = {
        "messages": [HumanMessage(content=user_query)],
        "accumulated_context": [],
        "agent_action": "search",
        "search_query": "",
        "candidate_answer": "",
        "critique_feedback": "",
        "loop_count": 0,
        "final_output": ""
    }
    
    # Run the graph
    result = compiled_graph.invoke(inputs, {"recursion_limit": 30})
    
    print("\n" + "=" * 80)
    print("FINAL VERIFIED dossier")
    print("=" * 80)
    print(result["messages"][-1].content)
    print("=" * 80)
    
    # Audit trail printout
    print("\nCONTEXT COLLECTION AUDIT TRAIL")
    print("=" * 80)
    for idx, ctx in enumerate(result.get("accumulated_context", []), 1):
        print(f"\n[Retrieval #{idx}]\n{ctx}")
    print("=" * 80 + "\n")
